# Step 3: Projection Model

Builds the per-player-per-week fantasy point projection model. Built up piece by piece, matching `ROADMAP.md` Step 3:
1. Multi-season raw data pull
2. Feature engineering (trailing form, season-to-date/prior-season, categorical)
3. Train/val/test split by season
4. Baseline model
5. LightGBM model
6. Evaluation (per position)


In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from nflverse_loader import load_players, load_weekly_stats
from data_loader import normalize_player_week


## 1. Multi-season raw data pull

2019-2024: recent enough that the modern passing-heavy game context still applies, enough volume for weekly-level training. Train 2019-2022, validate 2023, test 2024 (picked in piece 6).

Two real bugs surfaced and fixed while wiring this up (see `PROJECTS.md`/`ROADMAP.md` for detail): `PLAYERS_SCHEMA` was missing `season` (team/bye_week are season-dependent, not identity metadata — only worked before because every caller pulled one season at a time), and 2022's bye-week derivation double-counted the Week 17 Bills-Bengals game (suspended after Damar Hamlin's on-field cardiac arrest, never replayed) as a phantom bye for both teams.

In [2]:
SEASONS = list(range(2019, 2025))

players = load_players(SEASONS)
weekly = load_weekly_stats(SEASONS)
player_week = normalize_player_week(players, weekly)

print(f"players: {players.shape}, weekly: {weekly.shape}, player_week: {player_week.shape}")
player_week["season"].value_counts().sort_index()


players: (18576, 6), weekly: (39224, 14), player_week: (39224, 19)


season
2019    6165
2020    6354
2021    6702
2022    6653
2023    6640
2024    6710
Name: count, dtype: int64

In [3]:
# Sanity check: fantasy points by position, roughly matches expected scale
# (QB highest, then RB/WR, TE a bit lower, K always 0 — kicker scoring isn't
# wired into WEEKLY_STATS_SCHEMA yet, see schema.py).
player_week.groupby("position")["fantasy_points"].agg(["count", "mean"]).round(2)


,count,mean
position,,
DB,29,0.51
DL,1,0.00
K,3330,0.00
LB,9,0.11
P,15,0.00
QB,4065,14.02
RB,9397,8.00
TE,7280,5.63
WR,15098,7.61


## 2. Feature engineering: trailing form

Rolling averages of a player's own `fantasy_points`, computed causally (shifted so a given week's features never see that week's own result). Carries across season boundaries rather than resetting to NaN each September - a player's last 3 games are still the most relevant signal of current form.


In [4]:
from features import add_trailing_form_features

featured = add_trailing_form_features(player_week)
featured[["trailing_3g_avg", "trailing_5g_avg"]].describe()


,trailing_3g_avg,trailing_5g_avg
count,37916.000000,37916.000000
mean,7.468254,7.483758
std,6.749042,6.421094
min,-2.780000,-2.780000
25%,1.766667,2.120000
50%,5.833333,6.000000
75%,11.733333,11.740000
max,46.800000,46.800000


In [5]:
# Eyeball check against a real player's game log, including a season boundary
# and a real injury gap (McCaffrey missed 2024 weeks 1-9).
sample_id = "00-0033280"  # Christian McCaffrey
cols = ["name", "season", "week", "fantasy_points", "trailing_3g_avg", "trailing_5g_avg"]
featured.loc[featured["player_id"] == sample_id, cols].sort_values(["season", "week"]).tail(12)


,name,season,week,fantasy_points,trailing_3g_avg,trailing_5g_avg
30121,Christian McCaffrey,2023,13,22.3,24.133333,24.96
30435,Christian McCaffrey,2023,14,16.3,24.833333,24.90
30795,Christian McCaffrey,2023,15,41.7,23.166667,22.20
31180,Christian McCaffrey,2023,16,25.1,26.766667,26.50
31547,Christian McCaffrey,2023,17,13.1,27.700000,27.26
32367,Christian McCaffrey,2023,20,31.8,26.633333,23.70
32452,Christian McCaffrey,2023,21,29.2,23.333333,25.60
32493,Christian McCaffrey,2023,22,28.0,24.700000,28.18
35791,Christian McCaffrey,2024,10,16.7,29.666667,25.44
36121,Christian McCaffrey,2024,11,14.6,24.633333,23.76
